# 第4节:束搜索与序列生成

本节介绍序列生成任务中的解码策略,重点讲解束搜索(Beam Search)算法。

## 学习目标

1. 理解序列生成中的解码问题
2. 掌握贪心搜索的原理和局限性
3. 理解穷举搜索的完备性和计算代价
4. 深入理解束搜索算法的工作原理
5. 实现束搜索并应用于机器翻译
6. 学会调整束宽参数平衡精度和效率

## 4.1 序列生成的解码问题

### 问题定义

在Seq2Seq模型中,我们逐个预测输出序列,直到预测序列中出现特定的序列结束词元"&lt;eos&gt;"。

在任意时间步$t'$,解码器输出$y_{t'}$的概率取决于:
- 时间步$t'$之前的输出子序列 $y_1, \ldots, y_{t'-1}$
- 上下文变量 $\mathbf{c}$ (编码器的输出)

**符号定义**:
- $\mathcal{Y}$: 输出词表(包含"&lt;eos&gt;")
- $|\mathcal{Y}|$: 词表大小
- $T'$: 输出序列的最大词元数

**目标**: 从所有 $\mathcal{O}(|\mathcal{Y}|^{T'})$ 个可能的输出序列中寻找**理想的输出**。

### 挑战

搜索空间极大!

例如:
- 词表大小: $|\mathcal{Y}| = 10,000$
- 最大长度: $T' = 10$
- 可能序列数: $10,000^{10} = 10^{40}$ (天文数字!)

我们需要高效的搜索策略在**质量**和**速度**之间取得平衡。

In [ ]:
import torch
from torch import nn
import math
from d2l import torch as d2l
import matplotlib.pyplot as plt
import numpy as np

## 4.2 贪心搜索(Greedy Search)

### 策略

**贪心搜索**是最简单的解码策略:对于输出序列的每一时间步$t'$,选择具有**最高条件概率**的词元:

$$y_{t'} = \operatorname*{argmax}_{y \in \mathcal{Y}} P(y \mid y_1, \ldots, y_{t'-1}, \mathbf{c})$$

一旦输出序列包含了"&lt;eos&gt;"或者达到其最大长度$T'$,则输出完成。

### 示例

![贪心搜索示例](https://zh.d2l.ai/_images/s2s-prob1.svg)

假设输出中有四个词元"A","B","C"和"&lt;eos&gt;"。每个时间步下的四个数字分别表示在该时间步生成这四个词元的条件概率。

贪心搜索在每个时间步选择具有最高条件概率的词元:
- 时间步1: 选择"A" (0.5)
- 时间步2: 选择"B" (0.4)
- 时间步3: 选择"C" (0.4)
- 时间步4: 选择"&lt;eos&gt;" (0.6)

输出序列的条件概率: $0.5 \times 0.4 \times 0.4 \times 0.6 = 0.048$

In [ ]:
def greedy_search(decoder, encoder_output, src_valid_len, max_len, device):
    """贪心搜索实现
    
    参数:
        decoder: 解码器模型
        encoder_output: 编码器输出
        src_valid_len: 源序列有效长度
        max_len: 最大生成长度
        device: 设备
    
    返回:
        output_seq: 生成的序列
        scores: 每个时间步的概率
    """
    batch_size = encoder_output.shape[0]
    dec_state = decoder.init_state(encoder_output, src_valid_len)
    
    # 开始标记
    dec_X = torch.unsqueeze(
        torch.tensor([decoder.vocab['<bos>']] * batch_size, device=device), dim=1)
    
    output_seq, scores = [], []
    
    for _ in range(max_len):
        Y, dec_state = decoder(dec_X, dec_state)
        # Y形状: (batch_size, 1, vocab_size)
        
        # 选择概率最高的词元
        probs = torch.softmax(Y, dim=2)
        dec_X = probs.argmax(dim=2)  # 贪心选择
        pred = dec_X.squeeze(dim=1).type(torch.int32).item()
        
        output_seq.append(pred)
        scores.append(probs.max(dim=2)[0].item())
        
        # 如果遇到结束标记,停止
        if pred == decoder.vocab['<eos>']:
            break
    
    return output_seq, scores

### 贪心搜索的问题

**贪心搜索无法保证得到最优序列!**

![贪心搜索的局限](https://zh.d2l.ai/_images/s2s-prob2.svg)

在上图的另一个例子中:
- 如果在时间步2选择"C"(第二高概率)而不是"B"
- 最终序列"A","C","B","&lt;eos&gt;"的概率是 $0.5 \times 0.3 \times 0.6 \times 0.6 = 0.054$
- 这**大于**贪心搜索得到的 $0.048$!

**原因**: 贪心搜索只看局部最优,忽略了全局最优。

### 计算复杂度

贪心搜索的计算量: $\mathcal{O}(|\mathcal{Y}| \cdot T')$

例如: $|\mathcal{Y}| = 10,000, T' = 10$
- 需要评估: $10,000 \times 10 = 10^5$ 次
- 非常快! 但质量有限

## 4.3 穷举搜索(Exhaustive Search)

### 策略

如果目标是获得**最优序列**,我们可以使用**穷举搜索**:
- 穷举地列举所有可能的输出序列及其条件概率
- 选择条件概率最高的序列

### 最优性

穷举搜索保证找到使下式最大的序列:

$$\prod_{t'=1}^{T'} P(y_{t'} \mid y_1, \ldots, y_{t'-1}, \mathbf{c})$$

这是基于输入序列生成输出序列的**真实条件概率**。

### 计算复杂度

计算量: $\mathcal{O}(|\mathcal{Y}|^{T'})$

例如: $|\mathcal{Y}| = 10,000, T' = 10$
- 需要评估: $10,000^{10} = 10^{40}$ 个序列
- **计算上不可行!**

### 比较

| 方法 | 精度 | 计算量 | 可行性 |
|------|------|--------|--------|
| 贪心搜索 | 低 | $\mathcal{O}(|\mathcal{Y}| \cdot T')$ | ✅ 可行 |
| 穷举搜索 | 最高 | $\mathcal{O}(|\mathcal{Y}|^{T'})$ | ❌ 不可行 |
| **束搜索** | **中等** | **$\mathcal{O}(k \cdot |\mathcal{Y}| \cdot T')$** | **✅ 可行** |

**需要一个折中方案!** → **束搜索**

## 4.4 束搜索(Beam Search)

### 核心思想

**束搜索**是贪心搜索的改进版本,在精度和计算代价之间取得平衡。

**关键参数**: 束宽(beam size) $k$

### 算法流程

1. **时间步1**: 选择具有最高条件概率的 $k$ 个词元
   - 这$k$个词元将分别是$k$个候选输出序列的第一个词元

2. **后续时间步**: 基于上一时间步的$k$个候选输出序列
   - 从 $k \cdot |\mathcal{Y}|$ 个可能的选择中
   - 挑出具有最高条件概率的$k$个候选输出序列

3. **终止**: 当所有候选序列都包含"&lt;eos&gt;"或达到最大长度

4. **选择**: 从最终的$k$个候选中选择概率最高的序列

### 示例

![束搜索过程(束宽=2)](https://zh.d2l.ai/_images/beam-search.svg)

假设:
- 词表: $\mathcal{Y} = \{A, B, C, D, E\}$ (包含"&lt;eos&gt;")
- 束宽: $k = 2$
- 最大长度: $T' = 3$

**时间步1**:
- 选择概率最高的2个词元: $A$ 和 $C$

**时间步2**:
- 对于每个候选($A$, $C$),计算所有可能的扩展
- 计算 $P(A, y_2 \mid \mathbf{c})$ 和 $P(C, y_2 \mid \mathbf{c})$ 对所有 $y_2 \in \mathcal{Y}$
- 从这10个值中选择最大的2个: $P(A, B \mid \mathbf{c})$ 和 $P(C, E \mid \mathbf{c})$

**时间步3**:
- 继续扩展,计算 $P(A, B, y_3 \mid \mathbf{c})$ 和 $P(C, E, y_3 \mid \mathbf{c})$
- 从这10个值中选择最大的2个: $P(A, B, D \mid \mathbf{c})$ 和 $P(C, E, D \mid \mathbf{c})$

**最终候选序列**:
1. $A$
2. $C$
3. $A, B$
4. $C, E$
5. $A, B, D$
6. $C, E, D$

### 长度归一化

选择最终输出序列时,我们使用**长度归一化的分数**:

$$\text{score}(y_1, \ldots, y_L) = \frac{1}{L^\alpha} \log P(y_1, \ldots, y_L \mid \mathbf{c})$$

$$= \frac{1}{L^\alpha} \sum_{t'=1}^L \log P(y_{t'} \mid y_1, \ldots, y_{t'-1}, \mathbf{c})$$

其中:
- $L$: 最终候选序列的长度
- $\alpha$: 长度惩罚系数,通常设置为 $0.75$

**为什么需要长度归一化?**
- 较长的序列有更多的对数项相加
- 对数概率都是负数,相加会使总分更负
- 不归一化会偏向选择短序列
- $L^\alpha$ 用于惩罚长序列,平衡长短序列的分数

In [ ]:
def beam_search(decoder, encoder_output, src_valid_len, 
                beam_size, max_len, device, alpha=0.75):
    """束搜索实现
    
    参数:
        decoder: 解码器模型
        encoder_output: 编码器输出
        src_valid_len: 源序列有效长度  
        beam_size: 束宽
        max_len: 最大生成长度
        device: 设备
        alpha: 长度惩罚系数
    
    返回:
        best_seq: 最优序列
        best_score: 最优分数
    """
    batch_size = encoder_output.shape[0]
    assert batch_size == 1, "Beam search暂时只支持batch_size=1"
    
    vocab_size = len(decoder.vocab)
    dec_state = decoder.init_state(encoder_output, src_valid_len)
    
    # 初始化: k个候选序列
    # 每个候选: (sequence, score, decoder_state)
    bos_id = decoder.vocab['<bos>']
    eos_id = decoder.vocab['<eos>']
    
    # 开始标记
    candidates = [([bos_id], 0.0, dec_state)]
    completed = []  # 已完成的序列
    
    for step in range(max_len):
        all_candidates = []
        
        for seq, score, state in candidates:
            # 如果已经生成<eos>,加入完成列表
            if seq[-1] == eos_id:
                completed.append((seq, score, state))
                continue
            
            # 解码下一个词元
            dec_X = torch.tensor([[seq[-1]]], device=device)
            Y, new_state = decoder(dec_X, state)
            
            # 计算概率
            log_probs = torch.log_softmax(Y.squeeze(0), dim=-1)
            
            # 对每个可能的下一个词元
            for token_id in range(vocab_size):
                new_seq = seq + [token_id]
                new_score = score + log_probs[0, token_id].item()
                all_candidates.append((new_seq, new_score, new_state))
        
        # 选择top-k候选
        # 按分数排序
        all_candidates.sort(key=lambda x: x[1], reverse=True)
        candidates = all_candidates[:beam_size]
        
        # 如果所有候选都已完成,提前结束
        if len(candidates) == 0:
            break
    
    # 将未完成的候选也加入完成列表
    completed.extend(candidates)
    
    # 应用长度归一化,选择最优序列
    best_seq, best_score = None, float('-inf')
    for seq, score, _ in completed:
        # 移除<bos>
        seq_without_bos = seq[1:]
        length = len(seq_without_bos)
        
        # 长度归一化分数
        normalized_score = score / (length ** alpha)
        
        if normalized_score > best_score:
            best_score = normalized_score
            best_seq = seq_without_bos
    
    return best_seq, best_score

### 束搜索的特性

#### 1. 计算复杂度

$$\text{复杂度} = \mathcal{O}(k \cdot |\mathcal{Y}| \cdot T')$$

介于贪心搜索和穷举搜索之间。

例如: $k=5, |\mathcal{Y}|=10,000, T'=10$
- 需要评估: $5 \times 10,000 \times 10 = 5 \times 10^5$ 次
- 比贪心慢5倍,但质量更好
- 远快于穷举($10^{40}$)

#### 2. 束宽的影响

| 束宽$k$ | 精度 | 速度 | 适用场景 |
|---------|------|------|----------|
| $k=1$ | 低 | 最快 | 贪心搜索 |
| $k=5$ | 中 | 快 | 实时应用 |
| $k=10$ | 较高 | 中 | 标准设置 |
| $k=50$ | 高 | 慢 | 离线翻译 |
| $k=\infty$ | 最高 | 极慢 | 穷举搜索 |

**实践中**: $k \in [5, 10]$ 是常用的选择。

#### 3. 优势

- ✅ 比贪心搜索更准确
- ✅ 计算上可行
- ✅ 灵活调整精度-速度权衡
- ✅ 避免局部最优陷阱

#### 4. 局限

- ❌ 不保证全局最优
- ❌ 束宽$k$需要手动调整
- ❌ 对于某些任务(如对话),可能过于保守

## 4.5 束搜索可视化

让我们通过一个简单的例子可视化束搜索的过程。

In [ ]:
def visualize_beam_search():
    """
    可视化束搜索过程
    """
    # 模拟概率分布
    # 时间步1: 5个词的概率
    step1_probs = np.array([0.5, 0.3, 0.1, 0.05, 0.05])  # A, C, B, D, E
    tokens = ['A', 'C', 'B', 'D', 'E']
    
    # 时间步2: 给定前一个词,每个词的概率(简化)
    step2_probs = {
        'A': np.array([0.1, 0.2, 0.4, 0.2, 0.1]),  # A后面各词的概率
        'C': np.array([0.2, 0.1, 0.1, 0.2, 0.4])   # C后面各词的概率
    }
    
    beam_size = 2
    
    # 可视化
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 时间步1
    axes[0].bar(tokens, step1_probs, color='skyblue')
    axes[0].set_title('时间步1: 选择Top-2', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('概率', fontsize=12)
    axes[0].set_ylim([0, 0.6])
    
    # 标记选中的
    top2_idx = np.argsort(step1_probs)[-beam_size:]
    for idx in top2_idx:
        axes[0].bar(tokens[idx], step1_probs[idx], color='orange')
    
    axes[0].text(0.5, 0.55, f'选择: {tokens[0]}, {tokens[1]}', 
                ha='center', fontsize=12, color='red', fontweight='bold')
    
    # 时间步2: 扩展
    all_seqs = []
    all_probs = []
    
    for prev_token in ['A', 'C']:
        prev_prob = step1_probs[tokens.index(prev_token)]
        for i, next_token in enumerate(tokens):
            seq = f"{prev_token}-{next_token}"
            prob = prev_prob * step2_probs[prev_token][i]
            all_seqs.append(seq)
            all_probs.append(prob)
    
    # 排序并选择top-k
    sorted_indices = np.argsort(all_probs)[-beam_size:]
    
    axes[1].bar(range(len(all_seqs)), all_probs, color='lightblue')
    axes[1].set_xticks(range(len(all_seqs)))
    axes[1].set_xticklabels(all_seqs, rotation=45, ha='right')
    axes[1].set_title('时间步2: 从10个候选中选择Top-2', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('联合概率', fontsize=12)
    
    # 标记选中的
    for idx in sorted_indices:
        axes[1].bar(idx, all_probs[idx], color='orange')
    
    selected_seqs = [all_seqs[i] for i in sorted_indices]
    axes[1].text(5, max(all_probs) * 0.9, 
                f'选择: {selected_seqs[1]}, {selected_seqs[0]}',
                ha='center', fontsize=12, color='red', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n束搜索过程说明:")
    print("1. 时间步1: 从5个词中选择概率最高的2个(束宽=2)")
    print(f"   → 选择: A(0.5), C(0.3)")
    print("\n2. 时间步2: 扩展这2个候选,得到2×5=10个新候选")
    print("   计算每个候选的联合概率 P(w1, w2)")
    print(f"   → 选择概率最高的2个: {selected_seqs[1]}, {selected_seqs[0]}")
    print("\n3. 继续此过程,直到生成<eos>或达到最大长度")

# 运行可视化
visualize_beam_search()

## 4.6 束搜索 vs 贪心搜索对比实验

让我们通过一个模拟实验对比两种搜索策略的性能。

In [ ]:
def compare_search_strategies():
    """
    对比贪心搜索和束搜索
    """
    # 模拟一个简单的序列生成问题
    # 真实最优序列: A -> C -> B -> <eos>
    # 贪心会选择: A -> B -> C -> <eos>
    
    print("="*60)
    print("序列生成对比: 贪心搜索 vs 束搜索")
    print("="*60)
    
    # 场景设置
    print("\n场景: 生成长度为3的序列")
    print("词表: {A, B, C, <eos>}")
    print("\n时间步1概率:")
    print("  P(A) = 0.5 ← 贪心选择")
    print("  P(C) = 0.3")
    print("  P(B) = 0.2")
    
    print("\n时间步2概率 (给定时间步1):")
    print("  P(B|A) = 0.6 ← 贪心选择")
    print("  P(C|A) = 0.3")
    print("  P(C|C) = 0.7")
    
    print("\n时间步3概率:")
    print("  P(C|A,B) = 0.5")
    print("  P(B|A,C) = 0.8")
    print("  P(B|C,C) = 0.9")
    
    # 贪心搜索
    greedy_path = "A → B → C → <eos>"
    greedy_prob = 0.5 * 0.6 * 0.5 * 1.0
    
    print("\n" + "="*60)
    print("贪心搜索结果:")
    print("="*60)
    print(f"路径: {greedy_path}")
    print(f"概率: 0.5 × 0.6 × 0.5 = {greedy_prob:.4f}")
    
    # 束搜索 (k=2)
    beam_paths = [
        ("A → C → B → <eos>", 0.5 * 0.3 * 0.8),
        ("C → C → B → <eos>", 0.3 * 0.7 * 0.9),
        ("A → B → C → <eos>", greedy_prob)
    ]
    
    print("\n" + "="*60)
    print("束搜索结果 (beam_size=2):")
    print("="*60)
    print("\n考虑的所有候选路径:")
    for i, (path, prob) in enumerate(beam_paths, 1):
        print(f"{i}. {path}")
        print(f"   概率: {prob:.4f}")
    
    best_path, best_prob = max(beam_paths, key=lambda x: x[1])
    print(f"\n最优路径: {best_path}")
    print(f"最优概率: {best_prob:.4f}")
    
    # 对比
    improvement = (best_prob - greedy_prob) / greedy_prob * 100
    print("\n" + "="*60)
    print("对比结果:")
    print("="*60)
    print(f"贪心搜索概率: {greedy_prob:.4f}")
    print(f"束搜索概率:   {best_prob:.4f}")
    print(f"改进幅度:      {improvement:.1f}%")
    print("\n结论: 束搜索找到了更好的序列!")

# 运行对比
compare_search_strategies()

## 4.7 实践建议

### 束宽选择指南

```python
# 不同任务的推荐束宽
beam_size_recommendations = {
    '机器翻译': 5-10,      # 平衡质量和速度
    '文本摘要': 10-20,     # 需要更高质量
    '对话生成': 1-5,       # 需要多样性,束宽太大会太保守
    '语音识别': 10-50,     # 高质量要求
    '实时应用': 1-5,       # 速度优先
}
```

### 长度惩罚调整

```python
# alpha参数的影响
alpha_effects = {
    0.0: "不惩罚长度,倾向短序列",
    0.5: "轻度惩罚",
    0.75: "标准设置(推荐)",
    1.0: "强惩罚,倾向长序列",
    1.5: "很强惩罚,可能过长"
}
```

### 优化技巧

1. **批处理束搜索**:
   ```python
   # 将k个候选打包成batch,并行计算
   beam_states = torch.cat([state for _, _, state in candidates], dim=1)
   ```

2. **早停策略**:
   ```python
   # 如果已有足够多的完成序列,可以提前停止
   if len(completed) >= beam_size * 1.5:
       break
   ```

3. **多样性增强**:
   ```python
   # Diverse Beam Search: 惩罚相似候选
   # 鼓励不同的候选序列
   ```

### 调试技巧

```python
# 可视化束搜索过程
def debug_beam_search(candidates, step):
    print(f"\n时间步 {step}:")
    for i, (seq, score, _) in enumerate(candidates):
        tokens = [vocab.idx_to_token[idx] for idx in seq]
        print(f"候选 {i+1}: {' '.join(tokens)} (分数: {score:.4f})")
```

## 4.8 束搜索的变体

### 1. Diverse Beam Search

**问题**: 标准束搜索倾向于生成相似的序列

**解决**: 将束分成$G$组,每组内进行束搜索,组间鼓励多样性

```python
# 伪代码
def diverse_beam_search(beam_size, num_groups):
    group_size = beam_size // num_groups
    for group in range(num_groups):
        # 对当前组进行束搜索
        # 添加多样性惩罚项: 惩罚与其他组相似的候选
        pass
```

### 2. Stochastic Beam Search

**问题**: 束搜索过于确定性,缺乏随机性

**解决**: 不总是选择top-k,而是根据概率采样

```python
# 根据概率采样而非总是选择top-k
probs = F.softmax(scores / temperature, dim=-1)
sampled_indices = torch.multinomial(probs, beam_size)
```

### 3. Constrained Beam Search

**应用**: 需要生成包含特定词的序列

**方法**: 在搜索过程中强制包含约束词

```python
# 确保生成的序列包含required_tokens
def constrained_beam_search(required_tokens):
    # 跟踪每个候选已包含的约束词
    # 优先选择包含更多约束词的候选
    pass
```

## 小结

### 核心要点

1. **序列搜索策略对比**:

| 策略 | 复杂度 | 精度 | 适用场景 |
|------|--------|------|----------|
| 贪心搜索 | $\mathcal{O}(|\mathcal{Y}| \cdot T')$ | 低 | 实时、资源受限 |
| 束搜索 | $\mathcal{O}(k \cdot |\mathcal{Y}| \cdot T')$ | 中-高 | **大多数场景(推荐)** |
| 穷举搜索 | $\mathcal{O}(|\mathcal{Y}|^{T'})$ | 最高 | 不可行 |

2. **束搜索的优势**:
   - ✅ 避免贪心搜索的局部最优问题
   - ✅ 计算上可行(不像穷举搜索)
   - ✅ 可调整束宽平衡精度和速度
   - ✅ 长度归一化避免偏向短序列

3. **关键参数**:
   - **束宽** $k$: 5-10是常用选择
   - **长度惩罚** $\alpha$: 通常0.75
   - 根据具体任务调整

4. **实践建议**:
   - 从小的束宽开始,逐步增加
   - 观察精度和速度的权衡
   - 对于对话等任务,考虑采样方法
   - 使用批处理优化计算效率

### 数学公式回顾

**长度归一化分数**:
$$\text{score} = \frac{1}{L^\alpha} \sum_{t'=1}^L \log P(y_{t'} \mid y_1, \ldots, y_{t'-1}, \mathbf{c})$$

**计算复杂度**:
$$\mathcal{O}(k \cdot |\mathcal{Y}| \cdot T')$$

### 应用场景

- **机器翻译**: 标准应用,beam_size=5-10
- **文本摘要**: 需要高质量,beam_size=10-20  
- **图像描述**: beam_size=3-5
- **语音识别**: beam_size=10-50
- **对话生成**: beam_size=1-3(避免太保守)

## 练习

1. **理论题**: 我们可以把穷举搜索看作一种特殊的束搜索吗?为什么?
   - 提示: 考虑束宽$k$的极限情况

2. **实现题**: 实现一个简单的束搜索,应用于字符级语言模型
   - 对比不同束宽($k=1, 5, 10$)的生成质量
   - 测量生成时间

3. **实验题**: 在机器翻译任务中应用束搜索
   - 束宽如何影响BLEU分数?
   - 束宽如何影响翻译速度?

4. **分析题**: 长度惩罚系数$\alpha$的作用
   - $\alpha=0$会发生什么?
   - $\alpha=2$会发生什么?
   - 找到最优的$\alpha$值

5. **优化题**: 实现批处理束搜索
   - 将$k$个候选打包成batch
   - 对比单个处理和批处理的速度

6. **扩展题**: 实现Diverse Beam Search
   - 将束分成多个组
   - 添加多样性惩罚项
   - 观察生成结果的多样性

7. **应用题**: 在你的项目中集成束搜索
   - 选择合适的束宽
   - 调整长度惩罚
   - 评估改进效果